In [1]:
import os, sys, json, time, requests
from dotenv import load_dotenv

load_dotenv(override=True)
print("[OK] Environnement reinitialise avec Groq API")


[OK] Environnement reinitialise avec Groq API


# 🚀 Migration LLM : De HuggingFace à Groq API

## 1. Pourquoi cette Migration ?

Initialement, le système utilisait l'API Inference de **HuggingFace** pour générer les explications en langage naturel.
Cependant, l'utilisation de HuggingFace présentait des blocages critiques pour une utilisation commerciale :

1. **Quotas Trop Faibles & Pannes Quota (HTTP 402 / 429)** :
   - Le plan gratuit HuggingFace offre ~1 000 requêtes/mois.
   - Dès qu'un utilisateur parcourait quelques cartes produits, le quota mensuel était entièrement consommé en quelques secondes, forçant le système à basculer indéfiniment en mode fallback.
2. **Latence Élevée & Dépendance DNS** :
   - Temps de réponse de **3 à 8 secondes** par explication.
   - Problèmes fréquents de résolution DNS (`api-inference.huggingface.co`) selon les réseaux locaux / FAI.

---

## 2. Comparatif : HuggingFace vs Groq API Direct

| Critère | HuggingFace Inference API | Groq API Direct (LPU) |
|---|---|---|
| **Infrastructure** | GPU partagé classique | **LPU (Language Processing Unit)** dédié |
| **Latence Moyenne** | 3.0s à 8.0s (très lent) | **0.8s à 1.4s (ultra-rapide, < 1.5s)** |
| **Quota Gratuit** | ~1 000 req / **mois** | **1 000 req / JOUR** |
| **Stabilité Réseau** | Erreurs DNS / 402 fréquentes | **99.9% disponibilité via Groq Cloud** |
| **Modèle Sélectionné** | Llama-3.3 70B (lent) | **Llama 3.3 70B / Qwen 27B** |


# 🛠️ 2. Modifications Effectuées dans le Code

### A. Dans le fichier `.env`
- Désactivation du token legacy HuggingFace (`HF_TOKEN`).
- Ajout de la clé `GROQ_API_KEY` et configuration du modèle `LLM_MODEL=qwen/qwen3.8-27b` (ou `llama-3.3-70b-versatile`).

```env
# HuggingFace (legacy API - conservé en commentaire)
# HF_TOKEN=hf_...

# Groq API Configuration (Accès Direct LPU)
GROQ_API_KEY=gsk_...
LLM_MODEL=qwen/qwen3.8-27b
LLM_MAX_TOKENS=200
CACHE_TTL=86400
```

### B. Dans `src/services/explanation.py`
1. Remplacement de `_call_huggingface_api()` par `_call_groq_api()` :
   - Point d'entrée : `https://api.groq.com/openai/v1/chat/completions`
   - Détection de l'erreur `429` (Quota journalier atteint) pour déclencher le circuit breaker.
2. Redirection des fonctions `explain_suggestion()` et `explain_suggestion_detailed()` vers `_call_groq_api()`.


In [2]:
# Démonstration du code de l'appel Groq API
import sys, os, time
sys.path.insert(0, '..')
from src.services.explanation import _call_groq_api

prompt_demo = "Explique en 1 phrase simple pourquoi le produit REDMI 15C est recommande pour un reassort urgent."

t0 = time.perf_counter()
reponse = _call_groq_api(prompt_demo, model=os.getenv("LLM_MODEL", "qwen/qwen3.8-27b"), max_tokens=100)
elapsed = time.perf_counter() - t0

print(f"[Chrono] Latence Groq API : {elapsed:.2f} secondes")
print(f"Reponse LLM : {reponse}")


[Chrono] Latence Groq API : 0.97 secondes
Reponse LLM : Le REDMI 15C est recommandé pour un reassort urgent car il s'agit d'un modèle à très forte rotation et faible prix qui connaît une demande constante, rendant le risque de rupture de stock particulièrement critique pour le chiffre d'affaires.


# 🔍 3. Différence : Réponse LLM (Groq) vs Réponse Robotisée (Fallback)

Le système dispose d'un mécanisme de **résilience à deux étages** :

### A. La Réponse LLM Réelle (Groq)
- **Caractéristique** : Rédigée dynamiquement par l'IA à partir des données transmises dans le prompt.
- **Style** : Fluide, naturel, sans phrases à trous, avec déductions intelligentes.
- **Exemple réel obtenu via Groq** :
  > *"Ce client achète habituellement ce produit tous les 39 jours. Il est passé 131 jours depuis sa dernière commande, ce qui signifie qu'il est en retard de plus de 3 fois par rapport à son rythme habituel. De plus, les volumes commandés sont en hausse (de 40 à 60 unités)..."*

### B. La Réponse Robotisée (Fallback Python)
- **Caractéristique** : Générée localement par du code Python statique si l'API ou le réseau est indisponible.
- **Style** : Phrases à trous pré-formatées, rigides, identiques d'un produit à un autre.
- **Exemple du Fallback Python** :
  > *"Ce client a commandé ce produit 4 fois au total, ce qui témoigne d'un achat régulier et récurrent. La dernière commande remonte à 38 jours, soit 1.8 fois son rythme habituel de 21 jours."*

---

### Tableau Comparatif Synthetique

| Critère | Réponse LLM (Groq API) | Réponse Robotisée (Fallback Python) |
|---|---|---|
| **Origine** | Modèle de Langage (Groq LPU Cloud) | Code Python (`_rule_based_explanation`) |
| **Flexibilité** | Synthèse naturelle, déductions fluides | Formules à trous codées en dur |
| **Temps d'exécution** | ~1.0s à 1.5s | < 0.01s (instantané) |
| **Dépendance réseau** | Nécessite connexion Internet | 100% Offline (Fonctionne sans réseau) |
| **Usage dans l'App** | Explication détaillée (au clic modale) | Cartes par défaut (`_skip_llm=True`) & Secours |


In [3]:
# Test de validation de latence sur 5 requetes consecutives vers Groq
import sys, time, os
sys.path.insert(0, '..')
from src.services.explanation import _call_groq_api

latences = []
model_name = os.getenv("LLM_MODEL", "qwen/qwen3.8-27b")
print(f"=== BENCHMARK DE LATENCE GROQ EN DIRECT ({model_name}) ===")
for i in range(5):
    t0 = time.perf_counter()
    res = _call_groq_api("En une phrase, pourquoi recommander un produit en hausse ?", model=model_name, max_tokens=50)
    dt = time.perf_counter() - t0
    latences.append(dt)
    status = "OK" if res else "ECHEC"
    print(f"Requete {i+1}/5 : {dt:.2f}s | Statut : {status}")

avg_lat = sum(latences)/len(latences)
print(f"Latence Moyenne : {avg_lat:.2f}s (Objectif < 2.0s ATTEINT)")


=== BENCHMARK DE LATENCE GROQ EN DIRECT (qwen/qwen3.8-27b) ===
Requete 1/5 : 1.32s | Statut : OK
Requete 2/5 : 0.72s | Statut : OK
Requete 3/5 : 0.84s | Statut : OK
Requete 4/5 : 0.80s | Statut : OK
Requete 5/5 : 0.89s | Statut : OK
Latence Moyenne : 0.91s (Objectif < 2.0s ATTEINT)
